# `|corr|` prefix size

Does a prefix other than 150 beat k=150 on the same outer folds?

k=150 stays unless another prefix beats it by at least 0.01 outer-fold XGBoost R² and by more than the fold standard error. `all` is the uncullled matrix. A requested k past the `|r| ≥ 0.95` cull uses every surviving column. This notebook reads the saved run. It does not retrain.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from experiments.blocks import report

payload = json.loads((ROOT / "experiments/002/results/k_grid.json").read_text())
verdict = payload["verdict"]
rows = []
for name, row in verdict["table"].items():
    gap = verdict["challengers"].get(name)
    rows.append({
        "k": name,
        "p": int(pd.Series(row["k_effective"]).median()),
        "ridge": row["ridge"]["r2_mean"],
        "xgb": row["xgb"]["r2_mean"],
        "svr": row["svr"]["r2_mean"],
        "xgb_rmse": row["xgb"]["rmse_mean"],
        "vs_150": None if gap is None else gap["mean"],
    })
scores = pd.DataFrame(rows)
scores["_ord"] = scores["k"].map(lambda n: 10**9 if n == "all" else int(n))
scores = scores.sort_values("_ord").drop(columns="_ord")
print(f"keep {verdict['keep']}")
report(scores, "Outer-fold mean R²", bar="xgb")


keep 150


k,p,ridge,xgb,svr,xgb_rmse,vs_150,
25,25,0.282,0.454,0.416,7.161,-0.089,
50,50,0.310,0.493,0.431,6.899,-0.049,
75,75,0.329,0.523,0.444,6.685,-0.020,
100,100,0.324,0.531,0.438,6.635,-0.012,
150,150,0.289,0.543,0.424,6.549,,
200,200,0.225,0.545,0.421,6.531,0.003,
300,300,0.099,0.520,0.373,6.709,-0.023,
400,400,-0.019,0.493,0.335,6.899,-0.050,
600,600,-0.477,0.501,0.286,6.838,-0.041,
825,822,-2.649,0.505,0.246,6.807,-0.037,


## Folds

Same five splits for every prefix.


In [ ]:
fold_rows = []
for name in scores["k"]:
    rec = {"k": name}
    for i, value in enumerate(verdict["table"][name]["xgb"]["r2_folds"], start=1):
        rec[f"f{i}"] = value
    fold_rows.append(rec)
report(pd.DataFrame(fold_rows), "XGBoost R² on each fold")


k,f1,f2,f3,f4,f5
25,0.462,0.450,0.396,0.512,0.448
50,0.509,0.485,0.459,0.525,0.490
75,0.541,0.472,0.496,0.590,0.515
100,0.552,0.502,0.485,0.577,0.537
150,0.556,0.526,0.494,0.597,0.540
200,0.546,0.555,0.484,0.593,0.549
300,0.558,0.514,0.445,0.577,0.505
400,0.510,0.476,0.426,0.541,0.509
600,0.511,0.473,0.447,0.561,0.514
825,0.546,0.487,0.422,0.567,0.505
